In [200]:
import pandas as pd
import ntpath
from openpyxl import load_workbook

#============================================================
#1. Data Extraction and Dilution Factor Correction (10X)
#============================================================

# 1. Extract raw GC data
def extract_GC_data(input_files):
    # Read the Excel file 
    xls = pd.ExcelFile(input_files, engine='openpyxl')
    sheet_names = xls.sheet_names
    sheet_names = [sheet for sheet in sheet_names if sheet != "Graph"]
    
    # Dictionary to store the data
    output_data = {
        "sampleID": [],
        "Butanol": [],
        "Acetic Acid": [],
        "Butyric Acid": [],
        "Hexanoic Acid": [],
    }

    # Product
    products = {
        "Butanol",
        "Acetic Acid",
        "Butyric Acid",
        "Hexanoic Acid"
    }
    
    # Scan for product headers
    for sheet in sheet_names:
#        print(f"Checking sheet: {sheet}")
        df = pd.read_excel(input_file, sheet_name=sheet, engine="openpyxl", header=None)
        if df.empty:
            continue
#        print(f"DataFrame shape: {df.shape}")
#        print(len(sheet_names))
#        display(df)


        
        # Start to scan
        i = 0
#        print(f"conc_results: {conc_results}")
#        print(f"Dataframe size is: {len(df)}")
        for i in range(len(df)):
            if pd.notna(df.iloc[i,0]) == True:
                sample_id = str(df.iloc[i, 0]).strip()                       # sampleID in A column
                sample_id = sample_id[:-4]                                   # Remove automatic sample numbering
                output_data["sampleID"].append(sample_id)                    # Add the sampleID to the dictionary
#                print(f"Initial: output_data = {output_data}")
#                print(f"Initial: key = {key}, i ={i}")
                for key in products:
                    output_data[key].append(0)
#                    print(f"Initialize output_data with 0: output_data = {output_data}")
#                    print(f"Initialize output_data with 0: key = {key}, i ={i}")
            for key in products:
                prod = str(df.iloc[i, 1])                                    # Column B: Product name
                conc = float(df.iloc[i, 3]) * 10                             # Column D: Concentration of product, 10X Dilution factor
                if key == prod:
                    output_data[key][-1] = conc 
#                    print(f"If: output_data = {output_data}")
#                    print(f"If: key = {key}, i ={i}")
                else:
 #                   print(f"Else; output_data is:{output_data}")
 #                   print(f"Else; key, and i is: {key}, {i}")
                    continue
                    
    # Convert the dictionary to a DataFrame
    result_df = pd.DataFrame(output_data)
    return result_df
    #display(result_df.head(10))
    # Add the result sheet
#    with pd.ExcelWriter(input_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
#         result_df.to_excel(writer, sheet_name=output_sheet_name, index=False)

In [201]:
#========================================
#2. sampleID Parsing and Data Processing
#========================================
def parse_sample_id(sample_id: str):
    sample_id = str(sample_id).strip()
    parts = sample_id.rsplit("-", 2)                  # Split from the right at up to 2 "-" delimiters (max 3 parts)
    if len(parts) != 3:
        print ("Please double check the sample ID")
        return None, None, None
    condition_name = parts[0]                         # Strain/Condition name
    time_part = parts[1]
    time_digits = re.findall(r"\d+", time_part)       # Extract only the number in time_part
    if not time_digits:
        print ("Please double check the sampling time")
        return None, None, None
    time_val = int(time_digits[0])                    # Use the first number in time_digits
    try:
        batch_num = int(parts[2])
    except ValueError:
        print ("Please double check the batch number")
        return None, None, None
    return condition_name, time_val, batch_num

#============================================================
#3. Graph Data Organization
#============================================================

def process_for_graph(input_file, result_df):
    # Copy the concentration DataFrame
    conc_df = result_df.copy()
    
    # Initialize parsed columns
    conc_df["parsed_condition"] = None
    conc_df["parsed_time"] = None
    conc_df["parsed_batch"]  = None
    
    # Parse sample IDs
    for idx, row in conc_df.iterrows():
        sID = str(row["sampleID"])
        condition_name, time_val, batch_num = parse_sample_id(sID)
        conc_df.loc[idx, "parsed_condition"] = condition_name
        conc_df.loc[idx, "parsed_time"] = time_val
        conc_df.loc[idx, "parsed_batch"]  = batch_num

    # Exclude None values
    conc_df = conc_df[conc_df["parsed_condition"].notna() &
            conc_df["parsed_time"].notna() &
            conc_df["parsed_batch"].notna()].copy()

    # Concentration columns
    data_cols = [
        "Butanol",
        "Acetic Acid",
        "Butyric Acid",
        "Hexanoic Acid"
    ]

    # Initialize graph data
    graph = []

    # Group by condition and batch number
    for (condition_name, batch_num), group_df in conc_df.groupby(["parsed_condition", "parsed_batch"], sort=False):
        graph.append([f"{condition_name} - Batch {batch_num}"])

        # Column headers
        col_header = ["Time"] + data_cols
        graph.append(col_header)

        # Sort by sampling time
        group_df = group_df.sort_values("parsed_time")

        # Add concentration data
        for idx, row in group_df.iterrows():
            row_list = [row["parsed_time"]]
            for c in data_cols:
                row_list.append(row[c])
            graph.append(row_list)

        # Add an empty row
        graph.append([])

    # Convert the graph data to a DataFrame
    graph_df = pd.DataFrame(graph)

    # Add the Graph worksheet
    with pd.ExcelWriter(input_file, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
        graph_df.to_excel(writer, sheet_name="Graph", index=False, header=False)

    return graph_df

In [ ]:
#============================================================
#4. Execution
#============================================================
input_files = ["Test.xlsx"]

for input_file in input_files:
    result_df = extract_GC_data(input_file)
    graph_df = process_for_graph(input_file, result_df)

    print(f"Graph worksheet created in: {input_file}")
